[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/21-regressao-multipla/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/21-regressao-multipla")
    print("Material preparado em:", Path.cwd())


# Regressão linear múltipla

Material de apoio — Aula 21

## Objetivos

Este guia formula regressão múltipla em notação matricial, implementa o
ajuste e interpreta seus coeficientes. Também discute interações,
colinearidade, incerteza dos coeficientes e comparação entre modelos.

## Como estudar este capítulo

A regressão múltipla amplia a reta simples para situações em que várias
características ajudam a explicar a resposta. A novidade mais importante
não é apenas adicionar colunas à matriz: a interpretação passa a ser
**condicional**. O coeficiente de idade descreve a variação associada à
idade quando os demais atributos do modelo são mantidos fixos.

O capítulo começa pela construção da matriz de projeto, porque cada
coluna corresponde a uma escolha de representação: variável numérica,
indicador de categoria ou interação. Depois ajustamos o modelo,
interpretamos os coeficientes e investigamos dois desafios: interações,
nas quais o efeito de uma variável depende de outra, e colinearidade,
que dificulta separar contribuições individuais.

Use a notação matricial como um mapa compacto. Cada linha de $X$
representa uma observação, cada coluna um atributo e $\beta$ contém um
coeficiente por coluna. O produto $X\beta$ produz todas as médias
previstas de uma vez. A intuição continua sendo a mesma da regressão
simples.

## Base de dados de apoio

Usaremos [Medical Insurance
Cost](https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset),
do Kaggle. Cada linha representa uma pessoa segurada.

In [ ]:
from pathlib import Path
from statistics import NormalDist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(20260827)
df = pd.read_csv(Path("../16-correlacao/data/insurance.csv"))
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

> **Interpretação**
>
> A base combina variáveis quantitativas e categóricas. `charges` é
> assimétrica, tabagismo separa patamares e as regiões possuem quatro
> níveis. Essas propriedades determinam a codificação e orientam o
> diagnóstico.

## Modelo estatístico

Para $i=1,\ldots,n$,

$$
Y_i=\beta_0+\sum_{j=1}^p\beta_jx_{ij}+\varepsilon_i,
\qquad E[\varepsilon_i\mid X_i]=0.
$$

$i$ indexa observações, $j$ indexa preditores, $x_{ij}$ é o valor do
preditor $j$ para a observação $i$ e $\beta_j$ é seu coeficiente
populacional.

$\beta_j$ compara a média prevista após aumentar $X_j$ em uma unidade,
mantendo fixos os demais preditores do modelo.

> **Interpretação**
>
> “Manter fixo” define uma comparação condicional, não uma intervenção.
> Variáveis omitidas, erro de mensuração e forma funcional inadequada
> ainda podem impedir uma leitura causal.

## Codificação da matriz de projeto

Usaremos `northeast` como referência regional e `female`, `no smoker`
como referências dos indicadores binários.

Como `region` possui quatro níveis, o modelo com intercepto precisa de
apenas três indicadores regionais. Para a parte regional do modelo,

$$
E[Y_i\mid X_i]
=\beta_0+\beta_{NW}D_{NW,i}
+\beta_{SE}D_{SE,i}
+\beta_{SW}D_{SW,i}
+z_i^T\gamma,
$$

em que $z_i^T\gamma$ reúne os demais preditores. Para uma observação de
`northeast`, os três indicadores regionais valem zero. Assim, cada
coeficiente regional mede uma diferença em relação a `northeast`,
mantendo os demais preditores fixos.

Se incluíssemos também $D_{NE}$, teríamos

$$
1=D_{NE,i}+D_{NW,i}+D_{SE,i}+D_{SW,i}.
$$

Portanto, a coluna do intercepto seria combinação linear exata das
quatro colunas indicadoras. A remoção de uma categoria elimina essa
dependência linear e permite estimar os coeficientes de forma única.

In [ ]:
Xdf = pd.DataFrame({
    "age": df["age"],
    "bmi": df["bmi"],
    "children": df["children"],
    "smoker_yes": (df["smoker"] == "yes").astype(int),
    "sex_male": (df["sex"] == "male").astype(int),
})
Xdf = pd.concat([
    Xdf,
    pd.get_dummies(df["region"], prefix="region", drop_first=True, dtype=int)
], axis=1)
Xdf.head()

> **Interpretação**
>
> Remover uma categoria evita dependência linear perfeita entre o
> intercepto e os indicadores. Alterar a referência muda a
> parametrização, mas não muda os valores ajustados.

## Forma matricial

Com uma coluna de 1 para o intercepto,

$$E[y\mid X]=X\beta,$$

em que $X$ é $n\times(p+1)$, $\beta$ é $(p+1)\times1$ e $y$ é
$n\times1$.

In [ ]:
X = np.column_stack([np.ones(len(df)), Xdf.to_numpy(float)])
y = df["charges"].to_numpy(float)
names = ["intercepto"] + list(Xdf.columns)
pd.Series({"linhas_X": X.shape[0], "colunas_X": X.shape[1],
           "tamanho_y": len(y), "posto_X": np.linalg.matrix_rank(X)})

O posto igual ao número de colunas confirma que os coeficientes estão
identificados nesta codificação.

## Perda e gradiente

$$J(b)=\|y-Xb\|_2^2=(y-Xb)^T(y-Xb).$$

Para expandir a perda, começamos transpondo a primeira parcela. Como
$(Xb)^T=b^TX^T$,

$$
(y-Xb)^T=y^T-b^TX^T.
$$

Em seguida, aplicamos a distributividade:

$$
\begin{aligned}
J(b)
&=(y^T-b^TX^T)(y-Xb)\\
&=y^Ty-y^TXb-b^TX^Ty+b^TX^TXb.
\end{aligned}
$$

Os dois termos cruzados são iguais. De fato, $y^TXb$ é um escalar e,
portanto, é igual ao seu transposto:

$$
y^TXb=(y^TXb)^T=b^TX^Ty.
$$

Assim,

$$J(b)=y^Ty-2b^TX^Ty+b^TX^TXb.$$

O primeiro termo, $y^Ty$, não depende de $b$. O segundo é linear em $b$
e o terceiro é quadrático em $b$. Essa forma deixa direta a derivação do
gradiente.

Para o termo quadrático, usamos a identidade

$$
\nabla_b(b^TAb)=(A+A^T)b.
$$

O vetor $b$ aparece duas vezes na forma quadrática. A derivada associada
ao $b^T$ da esquerda contribui com $Ab$, enquanto a derivada associada
ao $b$ da direita contribui com $A^Tb$. O gradiente reúne essas duas
contribuições.

### Exemplo com duas variáveis

Considere

$$
b=
\begin{bmatrix}b_1\\b_2\end{bmatrix},
\qquad
A=
\begin{bmatrix}
a_{11}&a_{12}\\
a_{21}&a_{22}
\end{bmatrix}.
$$

Expandindo a forma quadrática,

$$
b^TAb
=a_{11}b_1^2
+(a_{12}+a_{21})b_1b_2
+a_{22}b_2^2.
$$

As derivadas parciais são

$$
\frac{\partial(b^TAb)}{\partial b_1}
=2a_{11}b_1+(a_{12}+a_{21})b_2,
$$

$$
\frac{\partial(b^TAb)}{\partial b_2}
=(a_{12}+a_{21})b_1+2a_{22}b_2.
$$

Reunindo as derivadas no gradiente,

$$
\nabla_b(b^TAb)
=
\begin{bmatrix}
2a_{11}&a_{12}+a_{21}\\
a_{12}+a_{21}&2a_{22}
\end{bmatrix}
b
=(A+A^T)b.
$$

Neste problema, $A=X^TX$. Como $(X^TX)^T=X^TX$, a matriz é simétrica e
as duas parcelas são iguais. Portanto,

$$
\nabla_b(b^TX^TXb)=2X^TXb.
$$

Logo,

$$\nabla_bJ(b)=2X^T(Xb-y).$$

O componente $j$ é $2\sum_i x_{ij}(\widehat y_i-y_i)$: diferenças entre
previsão e valor observado, ponderadas pela coluna $j$.

### Jacobiana de uma função vetorial

Para uma função com $m$ entradas e $n$ saídas,

$$
h:\mathbb R^m\to\mathbb R^n,
\qquad
h(u)=
\begin{bmatrix}
h_1(u)\\ \vdots\\ h_n(u)
\end{bmatrix},
$$

a **Jacobiana** reúne todas as derivadas parciais:

$$
Dh(u)=
\left[\frac{\partial h_i(u)}{\partial u_j}\right]_{i,j}
\in\mathbb R^{n\times m}.
$$

A linha $i$ mostra como a saída $h_i$ responde às entradas. A coluna $j$
mostra como todas as saídas respondem à entrada $u_j$. Para uma pequena
variação $\Delta u$, a Jacobiana fornece a aproximação linear

$$
h(u+\Delta u)\approx h(u)+Dh(u)\Delta u.
$$

Aqui, $D$ é o operador de derivação. Assim, $Dh(u)$ significa “Jacobiana
da função $h$ no ponto $u$”.

### Jacobianas da previsão e dos resíduos

A previsão conjunta é a função vetorial

$$g(b)=Xb\in\mathbb R^n.$$

Como $g:\mathbb R^{p+1}\to\mathbb R^n$, sua derivada é uma matriz: a
**Jacobiana**

$$
Dg(b)
=\left[\frac{\partial g_i(b)}{\partial b_j}\right]_{i,j}
=X.
$$

O vetor de resíduos candidatos é $e(b)=y-g(b)$. Para manter o gradiente
na forma **previsão menos observado**, defina $d(b)=g(b)-y=Xb-y=-e(b)$.
Como elevar ao quadrado remove o sinal, $J(b)=e(b)^Te(b)=d(b)^Td(b)$, e
$Dd(b)=X$. Aplicando a regra da cadeia,

$$
\nabla J(b)=2\,Dd(b)^Td(b)=2X^T(Xb-y).
$$

> **Gradiente e Jacobiana não são o mesmo objeto**
>
> $Dg(b)$ é uma matriz $n\times(p+1)$ porque $g$ é vetorial. Já
> $\nabla J(b)$ é um vetor $(p+1)\times1$ porque $J$ é escalar.
> Reservaremos $J$ para a função objetivo, $Dg(b)$ para a Jacobiana da
> previsão e $Dd(b)$ para a Jacobiana da discrepância $d(b)=g(b)-y$.

## Ajuste numérico

Usamos `lstsq`, que resolve o problema sem formar explicitamente
$(X^TX)^{-1}$.

In [ ]:
beta = np.linalg.lstsq(X, y, rcond=None)[0]
coef = pd.Series(beta, index=names, name="coeficiente")
coef.round(2)

> **Interpretação**
>
> Mantendo os demais preditores fixos, um ano adicional de idade
> corresponde a cerca de US\$ 256,86, uma unidade de IMC a US\$ 339,19 e
> ser fumante a aproximadamente US\$ 23.848,54 na média prevista. Essas
> são associações condicionais do modelo.

### Ajuste completo por gradiente descendente

A função abaixo minimiza o erro quadrático médio. Dividir a SSE por $n$
não altera o ponto de mínimo, mas torna a escala do gradiente menos
dependente do tamanho da amostra.

In [ ]:
def regressao_gd(X, y, learning_rate=0.01, max_iter=50_000, tol=1e-6):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape

    beta = np.zeros(p)
    historico = []

    for _ in range(max_iter):
        prediction = X @ beta
        difference = prediction - y
        historico.append(np.mean(difference**2))

        gradient = (2/n) * X.T @ difference
        if np.linalg.norm(gradient) < tol:
            break

        beta = beta - learning_rate * gradient

    return beta, np.asarray(historico)

A primeira coluna de `X` continua sendo o intercepto. Padronizamos as
demais colunas para melhorar a geometria da otimização.

In [ ]:
X_gd = X.copy()
media = X_gd[:, 1:].mean(axis=0)
desvio = X_gd[:, 1:].std(axis=0)
X_gd[:, 1:] = (X_gd[:, 1:] - media) / desvio

beta_gd, historico_gd = regressao_gd(X_gd, y)
prediction_gd = X_gd @ beta_gd

pd.Series({
    "iterações": len(historico_gd),
    "MSE inicial": historico_gd[0],
    "MSE final": historico_gd[-1],
    "RMSE final": np.sqrt(np.mean((y - prediction_gd)**2)),
}).round(4)

In [ ]:
mse_ols = np.mean((y - X @ beta)**2)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(historico_gd / 1e6, linewidth=2.5,
        label="gradiente descendente")
ax.axhline(mse_ols / 1e6, color="darkorange", linestyle="--",
           linewidth=2, label="mínimo por OLS")
ax.set(xlabel="iteração", ylabel="MSE (milhões de US$²)")
ax.legend()
plt.tight_layout()
plt.show()

> **Interpretação**
>
> Todas as coordenadas de `beta` são atualizadas simultaneamente com o
> mesmo vetor gradiente. A curva cai rapidamente no início e se
> estabiliza no mesmo MSE alcançado pelo ajuste de OLS. A padronização
> altera a escala dos coeficientes, mas não a capacidade do modelo de
> reproduzir os valores ajustados por OLS.

### SGD para regressão múltipla

No batch completo, cada atualização usa as $n$ linhas de $X$. No
gradiente descendente estocástico, a linha $x_i^T$ e a resposta $y_i$
produzem uma atualização imediata:

$$
e_i=y_i-x_i^T b,
\qquad
g_i(b)=-2x_i e_i,
\qquad
b\leftarrow b-\kappa g_i(b).
$$

Aqui, $x_i$ contém o intercepto e todos os preditores da pessoa $i$, $b$
contém todos os coeficientes e $\kappa$ é a taxa de aprendizado. Depois
de visitar as $n$ linhas uma vez, completamos uma época e embaralhamos a
ordem novamente.

In [ ]:
def regressao_sgd(X, y, learning_rate=0.0005, epochs=120, seed=20261111):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape

    beta = np.zeros(p)
    historico = []
    generator = np.random.default_rng(seed)

    for epoch in range(epochs):
        # Cada época apresenta todas as linhas em uma nova ordem.
        order = generator.permutation(n)

        for i in order:
            prediction_i = X[i] @ beta
            difference_i = prediction_i - y[i]

            # Para uma observação, o gradiente tem um componente por coluna de X.
            gradient = 2 * X[i] * difference_i

            # SGD atualiza todos os coeficientes após esta única observação.
            beta = beta - learning_rate * gradient

        # A perda completa não participa do passo; serve para acompanhar o ajuste.
        residual = y - X @ beta
        historico.append(np.mean(residual**2))

    return beta, np.asarray(historico)

beta_sgd, historico_sgd = regressao_sgd(X_gd, y)
pd.Series({
    "épocas": len(historico_sgd),
    "atualizações": len(historico_sgd) * len(y),
    "MSE final": historico_sgd[-1],
    "RMSE final": np.sqrt(historico_sgd[-1]),
}).round(4)

> **Interpretação**
>
> O SGD calcula um vetor com tantos componentes quanto há coeficientes,
> mas usa somente uma linha para estimá-lo. Cada passo é barato e
> ruidoso. O embaralhamento impede que uma ordenação fixa — por exemplo,
> fumantes seguidos de não fumantes — imponha repetidamente a mesma
> sequência de correções.

### Mini-batch para regressão múltipla

O mini-batch usa um subconjunto $B$ de $m$ linhas. Mantemos $b$ fixo
enquanto calculamos os resíduos do lote, tiramos a média de suas
contribuições e só então atualizamos os coeficientes:

$$
g_B(b)=-\frac{2}{m}X_B^T(y_B-X_Bb),
\qquad
b\leftarrow b-\kappa g_B(b).
$$

$X_B$ e $y_B$ contêm apenas as linhas do lote. Se o último lote da época
tiver menos de $m$ observações, o denominador deve ser sua quantidade
real de linhas.

In [ ]:
def regressao_minibatch(
    X, y, batch_size=64, learning_rate=0.01, epochs=120, seed=20261111
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape

    beta = np.zeros(p)
    historico = []
    generator = np.random.default_rng(seed)

    for epoch in range(epochs):
        # O embaralhamento muda a composição dos lotes em cada época.
        order = generator.permutation(n)

        for start in range(0, n, batch_size):
            batch_ids = order[start:start + batch_size]
            X_batch = X[batch_ids]
            y_batch = y[batch_ids]

            prediction = X_batch @ beta
            difference = prediction - y_batch

            # Média das contribuições das observações presentes neste lote.
            gradient = (2 / len(batch_ids)) * X_batch.T @ difference

            # Uma atualização por lote, sempre simultânea em todos os coeficientes.
            beta = beta - learning_rate * gradient

        residual = y - X @ beta
        historico.append(np.mean(residual**2))

    return beta, np.asarray(historico)

beta_minibatch, historico_minibatch = regressao_minibatch(X_gd, y)
pd.Series({
    "épocas": len(historico_minibatch),
    "lotes por época": int(np.ceil(len(y) / 64)),
    "MSE final": historico_minibatch[-1],
    "RMSE final": np.sqrt(historico_minibatch[-1]),
}).round(4)

> **Interpretação**
>
> O produto `X_batch.T @ residual` acumula simultaneamente as
> contribuições para intercepto, idade, IMC, número de filhos e todos os
> indicadores categóricos. O mini-batch reduz parte da oscilação do SGD
> sem esperar a leitura da base inteira para fazer uma atualização.

### Comparação com batch completo e OLS

Como os coeficientes de `beta_gd`, `beta_sgd` e `beta_minibatch`
correspondem à matriz padronizada `X_gd`, calculamos também a solução
OLS nessa mesma escala. Comparar diretamente esses coeficientes com os
do primeiro ajuste, feito na escala original, seria incorreto; os
valores ajustados, porém, são equivalentes.

In [ ]:
beta_ols_padronizado = np.linalg.lstsq(X_gd, y, rcond=None)[0]

comparacao_otimizadores = []
for metodo, coeficientes in {
    "OLS (lstsq)": beta_ols_padronizado,
    "batch completo": beta_gd,
    "mini-batch (m=64)": beta_minibatch,
    "SGD (m=1)": beta_sgd,
}.items():
    residual = y - X_gd @ coeficientes
    comparacao_otimizadores.append({
        "método": metodo,
        "MSE": np.mean(residual**2),
        "RMSE": np.sqrt(np.mean(residual**2)),
        "distância até OLS": np.linalg.norm(
            coeficientes - beta_ols_padronizado
        ),
    })

pd.DataFrame(comparacao_otimizadores).set_index("método").round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
epochs_to_compare = min(
    len(historico_sgd), len(historico_minibatch), len(historico_gd) - 1
)
data_passes = np.arange(1, epochs_to_compare + 1)

# historico_gd[0] é a perda antes da primeira atualização batch.
ax.plot(data_passes, historico_gd[1:epochs_to_compare + 1] / 1e6,
        label="batch completo", linewidth=2.4)
ax.plot(data_passes, historico_minibatch[:epochs_to_compare] / 1e6,
        label="mini-batch", linewidth=2)
ax.plot(data_passes, historico_sgd[:epochs_to_compare] / 1e6,
        label="SGD", linewidth=1.7, alpha=.85)
ax.axhline(mse_ols / 1e6, color="black", linestyle="--",
           linewidth=1.5, label="mínimo OLS")
ax.set(xlabel="passagens completas pela amostra",
       ylabel="MSE (milhões de US$²)")
ax.legend()
plt.tight_layout()
plt.show()

> **Interpretação**
>
> Os três algoritmos procuram o mesmo mínimo da perda quadrática. Batch
> completo produz uma trajetória suave; mini-batch faz mais atualizações
> por época com ruído moderado; SGD atualiza após cada pessoa e oscila
> mais. A distância até OLS mostra que “terminar uma época” não é
> critério suficiente de convergência: taxa, número de épocas e
> tolerância precisam ser avaliados junto com a curva da perda. O eixo
> horizontal compara a mesma quantidade de dados processados: uma
> passagem completa pela amostra em cada método.

## Valores ajustados e qualidade

In [ ]:
prediction = X @ beta
residual = y-prediction
sse = np.sum(residual**2)
sst = np.sum((y-y.mean())**2)
pd.Series({
    "R²": 1-sse/sst,
    "RMSE": np.sqrt(np.mean(residual**2)),
    "média_resíduos": residual.mean(),
}).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y, prediction, s=10, alpha=.35)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "--", color="darkorange")
axes[0].set(xlabel="observado", ylabel="ajustado")
axes[1].scatter(prediction, residual, s=10, alpha=.35)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="ajustado", ylabel="resíduo")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> O modelo múltiplo explica muito mais variação que idade isolada,
> sobretudo pela inclusão de tabagismo. Ainda há assimetria, valores
> extremos e variância residual desigual; mais $R^2$ não significa
> modelo plenamente adequado.

## Interações

Centralizamos o IMC em 30 para que o coeficiente de tabagismo compare os
grupos em um valor substantivamente plausível:

$$bmi_c=bmi-30.$$

Para permitir inclinações de IMC distintas por tabagismo:

$$
E[Y]=\beta_0+\beta_1bmi_c+\beta_2smoker
+\beta_3(bmi_c\times smoker)+\cdots.
$$

In [ ]:
X_inter = Xdf.copy()
X_inter["bmi_c"] = X_inter.pop("bmi") - 30
X_inter["bmi_c_x_smoker"] = (
    X_inter["bmi_c"] * X_inter["smoker_yes"]
)
Xi = np.column_stack([np.ones(len(df)), X_inter.to_numpy(float)])
bi = np.linalg.lstsq(Xi, y, rcond=None)[0]
coef_interacao = pd.Series(
    bi, index=["intercepto"]+list(X_inter.columns)
)
coef_interacao.loc[
    ["bmi_c", "smoker_yes", "bmi_c_x_smoker"]
].round(2)

As inclinações estimadas do IMC em cada grupo são:

In [ ]:
pd.Series({
    "não fumantes": coef_interacao["bmi_c"],
    "fumantes": (
        coef_interacao["bmi_c"]
        + coef_interacao["bmi_c_x_smoker"]
    ),
}, name="US$ por unidade de IMC").round(2)

A melhora no ajuste é substancial:

In [ ]:
prediction_inter = Xi @ bi
residual_inter = y - prediction_inter
sst = np.sum((y - y.mean())**2)

def resumo_ajuste(erro, numero_preditores):
    sse = np.sum(erro**2)
    r2 = 1 - sse / sst
    r2_ajustado = 1 - (1-r2) * (len(y)-1) / (
        len(y)-numero_preditores-1
    )
    return np.sqrt(np.mean(erro**2)), r2_ajustado

pd.DataFrame({
    "sem interação": resumo_ajuste(residual, X.shape[1]-1),
    "com interação": resumo_ajuste(
        residual_inter, Xi.shape[1]-1
    ),
}, index=["RMSE", "R² ajustado"]).T.round(3)

### Diagnóstico visual após a interação

Comparamos os mesmos diagnósticos usados anteriormente: QQ-plot dos
resíduos e resíduos versus valores ajustados.

In [ ]:
probabilidades = (np.arange(1, len(y) + 1) - 0.5) / len(y)
quantis_teoricos = np.array([
    NormalDist().inv_cdf(float(p)) for p in probabilidades
])

fig, axes = plt.subplots(2, 2, figsize=(11, 7),
                         sharex="col", sharey="col")

modelos = [
    ("Sem interação", prediction, residual, "#1f6f8b"),
    ("Com IMC × tabagismo", prediction_inter, residual_inter, "#d95f02"),
]

for linha, (nome, ajustado, erro, cor) in enumerate(modelos):
    residuos_ordenados = np.sort(erro)
    inclinacao, intercepto = np.polyfit(
        quantis_teoricos, residuos_ordenados, 1
    )

    axes[linha, 0].scatter(
        quantis_teoricos, residuos_ordenados,
        s=8, alpha=0.45, color=cor
    )
    axes[linha, 0].plot(
        quantis_teoricos,
        intercepto + inclinacao * quantis_teoricos,
        color="black"
    )
    axes[linha, 0].set_title(f"{nome}: QQ-plot")
    axes[linha, 0].set_ylabel("resíduos ordenados")

    axes[linha, 1].scatter(
        ajustado, erro, s=8, alpha=0.28, color=cor
    )
    axes[linha, 1].axhline(0, color="black", linestyle="--")
    axes[linha, 1].set_title(f"{nome}: resíduos × ajustados")
    axes[linha, 1].set_ylabel("resíduo")

axes[1, 0].set_xlabel("quantis teóricos")
axes[1, 1].set_xlabel("valor ajustado")
plt.tight_layout()
plt.show()

> **Interpretação**
>
> Com interação, a inclinação do IMC é US\$ 23,53 entre não fumantes. O
> termo `bmi_c_x_smoker`, US\$ 1.443,10, é o acréscimo na inclinação
> entre fumantes. Assim, a inclinação estimada para fumantes é US\$
> 1.466,63 por unidade de IMC, mantendo os demais preditores fixos. O
> RMSE diminui de forma clara, mas o QQ-plot e o gráfico de resíduos
> ainda mostram desvios das hipóteses do modelo. Termos principais devem
> permanecer para preservar a hierarquia.

### Por que manter os termos principais?

Considere retirar `smoker_yes`, mas preservar a interação. O modelo
passaria a ser

$$
E[Y\mid X]
=\beta_0+\beta_1bmi_c
+\beta_3(bmi_c\times smoker)+z^T\gamma.
$$

Quando BMI = 30, temos $bmi_c=0$. Nesse ponto, tanto o termo principal
do IMC quanto a interação desaparecem. O modelo seria obrigado a
satisfazer

$$
E[Y\mid smoker=0,\,bmi=30]
=E[Y\mid smoker=1,\,bmi=30],
$$

mantendo os demais preditores fixos. Essa igualdade não foi aprendida
com os dados; ela foi imposta pela exclusão do termo principal de
tabagismo. O princípio hierárquico evita introduzir esse tipo de
restrição sem uma justificativa substantiva.

## Colinearidade

Colinearidade ocorre quando colunas de $X$ são altamente relacionadas.
Ela pode inflar a incerteza dos coeficientes sem destruir
necessariamente a qualidade preditiva conjunta.

### Colinearidade perfeita e forte

Se $x_2=2x_1$, então

$$
\widehat y
=\beta_0+\beta_1x_1+\beta_2x_2
=\beta_0+(\beta_1+2\beta_2)x_1.
$$

Os dados conseguem identificar a combinação $\beta_1+2\beta_2$, mas não
separar os dois coeficientes. Por exemplo, $(\beta_1,\beta_2)=(4,1)$ e
$(2,2)$ produzem a mesma combinação, igual a 6. Nesse caso, $X^TX$ é
singular.

Quando $x_2\approx2x_1$, a redundância deixa de ser exata e $X^TX$ pode
ser invertível. Ainda assim, pequenas mudanças na amostra podem fazer um
coeficiente aumentar enquanto o outro diminui. A combinação usada nas
previsões muda pouco, mas os coeficientes individuais ficam instáveis.
Por isso, colinearidade é uma questão de grau, e não apenas a presença
ou ausência de redundância perfeita.

In [ ]:
numeric_corr = Xdf[["age", "bmi", "children"]].corr()
numeric_corr.round(3)

Uma forma diagnóstica é observar a sensibilidade dos coeficientes a
pequenas mudanças na amostra.

## Bootstrap do vetor de coeficientes

In [ ]:
B = 1000
boot = np.empty((B, X.shape[1]))
for b in range(B):
    idx = rng.integers(0, len(df), len(df))
    boot[b] = np.linalg.lstsq(X[idx], y[idx], rcond=None)[0]

summary = pd.DataFrame({
    "estimativa": beta,
    "q2.5": np.quantile(boot, .025, axis=0),
    "q97.5": np.quantile(boot, .975, axis=0),
}, index=names)
summary.round(2)

> **Interpretação**
>
> Cada intervalo mantém fixa a especificação do modelo e quantifica a
> variação sob reamostragem das pessoas observadas. Ele não incorpora
> incerteza sobre quais variáveis deveriam entrar nem corrige vieses do
> desenho observacional.

## Validação e complexidade

Adicionar um preditor a um modelo não aumenta a SSE de treinamento. O
novo modelo sempre pode reproduzir o anterior atribuindo coeficiente
zero à nova variável. Por isso,

$$
R^2=1-\frac{SSE}{SST}
$$

nunca diminui quando comparamos modelos ajustados aos mesmos dados e com
a mesma variável resposta. Isso não significa que todo novo preditor
seja útil: ele pode reduzir a SSE apenas por se adaptar ao ruído da
amostra.

O **$R^2$ ajustado** corrige essa comparação pelos graus de liberdade:

$$
R^2_{\mathrm{aj}}
=1-\frac{SSE/(n-p-1)}{SST/(n-1)}
=1-(1-R^2)\frac{n-1}{n-p-1},
$$

em que:

-   $n$ é o número de observações;
-   $p$ é o número de preditores, sem contar o intercepto;
-   $n-p-1$ são os graus de liberdade residuais;
-   $n-1$ são os graus de liberdade associados à variação total.

A fração $SSE/(n-p-1)$ é a variância residual estimada. Quando incluímos
um preditor, a SSE pode cair, mas o denominador também perde um grau de
liberdade. Assim:

-   se a queda da SSE for suficientemente grande, $R^2_{\mathrm{aj}}$
    aumenta;
-   se o novo preditor explicar muito pouco, $R^2_{\mathrm{aj}}$
    diminui;
-   em modelos muito fracos, o $R^2_{\mathrm{aj}}$ pode até ser
    negativo.

Mais colunas podem melhorar a aproximação, mas também aumentar
variância, instabilidade e risco de capturar ruído.

> **Interpretação**
>
> O $R^2$ ajustado ajuda a comparar modelos com números diferentes de
> preditores, mas continua sendo calculado nos dados de treinamento.
> Para avaliar generalização, precisamos usar validação ou dados de
> teste.

## Como interpretar o modelo sem se perder nos coeficientes

O modelo múltiplo é uma extensão da reta: continuamos descrevendo uma
média, mas agora com várias informações simultâneas. Um roteiro seguro
é:

1.  **Definir a resposta e a unidade observacional.** Neste exemplo,
    cada linha é uma pessoa e a resposta são suas despesas.
2.  **Escolher atributos disponíveis para a pergunta.** Idade, IMC e
    tabagismo entram por razões diferentes e possuem unidades
    diferentes.
3.  **Codificar categorias.** Uma categoria vira referência; os
    coeficientes das demais expressam diferenças em relação a ela.
4.  **Ajustar o modelo.** O algoritmo encontra a combinação de
    coeficientes que reduz os resíduos.
5.  **Interpretar um coeficiente por vez.** A leitura sempre inclui
    “mantendo os demais atributos do modelo constantes”.
6.  **Verificar se essa comparação é plausível.** Combinações raras de
    atributos podem levar a interpretações apoiadas por poucos dados.
7.  **Analisar resíduos e validação.** Um bom coeficiente isolado não
    substitui a avaliação do modelo completo.

### Variável categórica

Se `smoker=yes` é comparado com `smoker=no`, seu coeficiente representa
a diferença média prevista entre fumantes e não fumantes com a mesma
idade e o mesmo IMC, segundo o modelo. Isso é uma comparação
condicional, não automaticamente um efeito causal.

### Interação

Uma interação responde a perguntas como: “a relação entre IMC e despesas
muda entre fumantes e não fumantes?”. Ao incluí-la, passamos a ter uma
inclinação para o grupo de referência e uma correção dessa inclinação
para o outro grupo.

## Colinearidade em linguagem prática

Quando dois atributos trazem quase a mesma informação, o modelo pode
prever bem, mas ter dificuldade para decidir quanto do efeito atribuir a
cada coeficiente. Sinais podem mudar e erros-padrão podem aumentar.
Antes de retirar variáveis mecanicamente, volte à pergunta: precisamos
de previsão ou de interpretação separada dos coeficientes?

> **Leitura recomendada**
>
> Leia primeiro a previsão produzida para perfis concretos e só depois
> examine os coeficientes. Isso reduz o risco de interpretar um termo
> fora da escala, da categoria de referência ou das interações do
> modelo.

## Questões de revisão

1.  Explique as dimensões de $X$, $\beta$, $y$ e $\nabla J$.
2.  Derive $\nabla J=2X^T(X\beta-y)$ e explique por que o algoritmo
    subtrai esse vetor.
3.  Interprete um coeficiente indicador e uma interação.
4.  Por que mudar a categoria de referência não muda previsões?
5.  Como colinearidade afeta interpretação e previsão?
6.  Por que o $R^2$ ajustado pode diminuir quando adicionamos um
    preditor?

## Bibliografia

-   James et al., *An Introduction to Statistical Learning*, capítulo 3.
-   Faraway, *Linear Models with R*.
-   Gelman, Hill e Vehtari, *Regression and Other Stories*.
-   Fox, *Applied Regression Analysis and Generalized Linear Models*,
    regressão múltipla.